## **Medical Image Processing**
### Lab 3 - Deep Learning


In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ---------------------------------------------- Part #3 ----------------------------------------------

**1. Install useful libraries and U-Net definition**


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
# Show versioning of deep learning libraries
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

2.9.0+cpu False


In [ ]:
# Directory that contains all the data/script
current_dir = "/content/drive/MyDrive/..."


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """Applies two consecutive conv-batchnorm-relu layers"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self,
                 in_channels=3,
                 out_channels=1,
                 init_filters=64,
                 depth=4,
                 bilinear=True):
        super(UNet, self).__init__()
        self.depth = depth
        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        # Encoder
        filters = init_filters
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            in_channels = filters
            filters *= 2

        # Bottleneck
        self.bottleneck = DoubleConv(in_channels, filters)

        # Decoder
        for d in range(depth):
            filters //= 2
            if bilinear:
                up = nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                    nn.Conv2d(filters * 2, filters, kernel_size=1)
                )
            else:
                up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)
            self.up_layers.append(nn.ModuleDict({
                'up': up,
                'conv': DoubleConv(filters * 2, filters)
            }))

        # Output layer
        self.out_conv = nn.Conv2d(init_filters, out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.down_layers:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for i in range(self.depth):
            skip = skip_connections[-(i+1)]
            up = self.up_layers[i]['up'](x)
            if up.size() != skip.size():
                # Resize in case of odd size mismatch
                up = F.interpolate(up, size=skip.shape[2:])
            x = torch.cat([skip, up], dim=1)
            x = self.up_layers[i]['conv'](x)

        return self.out_conv(x)

**2. Load the configuration and weights of the trained U-Net model**

In [ ]:
import json
import torch

def load_model_from_checkpoint(checkpoint_dir, epoch_to_load):
    # Path to the JSON file with saved parameters
    params_path = os.path.join(checkpoint_dir, 'training_params.json')

    # Load parameters from JSON
    with open(params_path, 'r') as f:
        params = json.load(f)

    print("Loaded parameters:", params)

    # Create the model using the loaded parameters
    model = UNet(
        in_channels=params['in_channels'],
        out_channels=params['out_channels'],
        init_filters=params['init_filters'],
        depth=params['depth']
    )

    # Path to the checkpoint
    checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch_to_load}.pt")

    # Load model state
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Checkpoint for epoch {epoch_to_load} loaded from {checkpoint_path}")

    return model, params

In [ ]:
checkpoint_dir = os.path.join(current_dir,'unet_training_checkpoints')  # checkpoint directory
epoch_number = 19  # epoch to load

model, params = load_model_from_checkpoint(checkpoint_dir, epoch_number)
input_size = tuple(params["input_size"])

Loaded parameters: {'input_size': [1024, 1024], 'in_channels': 3, 'out_channels': 1, 'init_filters': 32, 'depth': 3, 'n_epochs': 20, 'batch_size': 4, 'learning_rate': 0.001, 'checkpoint_freq': 1}
Checkpoint for epoch 19 loaded from /content/drive/MyDrive/MIP/PROGETTO /DATASETORIGINALE/unet_training_checkpoints/model_epoch_19.pt


**3. Apply trained model to the test set**




PREPROCESSING

In [ ]:
import cv2
def ROI_detection(img):
    """
    Generates the Region of Interest (ROI) mask by combining
    Hough Circle Transform and Intensity Thresholding.
    """
    HOUGH_PARAMS = {
        'dp': 1.2, 'minDist': 100, 'param1': 50, 'param2': 30,
        'minRadius': 200, 'maxRadius': 700
    }

    img_height, img_width = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # A. Hough Transform (Circular detection)
    gray_blur = cv2.medianBlur(gray, 5)


    circles = cv2.HoughCircles(gray_blur, cv2.HOUGH_GRADIENT, **HOUGH_PARAMS)
    mask_circle = np.zeros((img_height, img_width), dtype=np.uint8)
    msg_hough = "Hough: NO"

    if circles is not None:
        # Round coordinates to integers
        circles_rounded = np.round(circles[0, :]).astype("int")
        x, y, r = circles_rounded[0]
        # Draw the detected circle on the mask (white filled circle)
        cv2.circle(mask_circle, (x, y), r, 255, -1)
        msg_hough = "Hough: OK"
    else:
        # If no circles are detected, set the entire mask to white (full image)
        mask_circle[:] = 255
        print("Circle not found, using the full image as mask.")

    # B. Intensity Thresholding
    INTENSITY_THRESHOLD = 45
    # Create a binary mask for pixels above the intensity threshold
    _, mask_thresh = cv2.threshold(gray, INTENSITY_THRESHOLD, 255, cv2.THRESH_BINARY)


    # C. Union (Combine Masks)
    # Combine the geometric circle mask with the intensity mask using bitwise OR
    final_mask = cv2.bitwise_or(mask_circle, mask_thresh)

    return final_mask

def color_normalization(img, mask):
    """
    Applies statistical color normalization (Reinhard-like method)
    aligning the image distribution to a target Mean and Standard Deviation.
    """
    # Target statistics (calculated on training dataset computed using function target_statistic attached)
    TARGET_MEAN = [74.57, 151.81, 158.29]
    TARGET_STD  = [34.50, 14.12, 14.01]


    # Convert to LAB color space for statistical processing
    img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype("float32")
    l, a, b = cv2.split(img_lab)

    # Safety check: if mask is empty, return original image
    if cv2.countNonZero(mask) == 0:
        return img

    # Compute mean and std dev of the source image (masked area only)
    (mean_src, std_src) = cv2.meanStdDev(img_lab, mask=mask)
    l_mean, a_mean, b_mean = mean_src.flatten()
    l_std, a_std, b_std = std_src.flatten()

    eps = 1e-5 # Epsilon to avoid division by zero
    valid_pixels = (mask > 0)

    # Apply color transfer formula: Dest = (Src - MeanSrc) * (StdTgt / StdSrc) + MeanTgt
    l[valid_pixels] = ((l[valid_pixels] - l_mean) * (TARGET_STD[0] / (l_std + eps))) + TARGET_MEAN[0]
    a[valid_pixels] = ((a[valid_pixels] - a_mean) * (TARGET_STD[1] / (a_std + eps))) + TARGET_MEAN[1]
    b[valid_pixels] = ((b[valid_pixels] - b_mean) * (TARGET_STD[2] / (b_std + eps))) + TARGET_MEAN[2]

    # Clip values to valid range [0, 255]
    l = np.clip(l, 0, 255)
    a = np.clip(a, 0, 255)
    b = np.clip(b, 0, 255)

    # Merge channels and convert back to RGB uint8
    res_lab = cv2.merge((l, a, b))
    res_rgb = cv2.cvtColor(res_lab.astype("uint8"), cv2.COLOR_LAB2RGB)

    # Apply the mask to the final output to clean up background artifacts
    res_rgb = cv2.bitwise_and(res_rgb, res_rgb, mask=mask)

    return res_rgb

def clahe(img_rgb, mask):
    """
    Applies CLAHE (Contrast Limited Adaptive Histogram Equalization)
    on the LAB color space (L channel) to enhance local contrast
    while preserving original color information.

    Args:
        img_rgb: 3-channel image (RGB format expected)
        mask: Binary mask (same height and width)
    """
    # 1. Convert from RGB to LAB color space
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)

    # 2. Split channels (L=Lightness, A and B = Color components)
    l, a, b = cv2.split(lab)

    # 3. Configure and apply CLAHE specifically on the L channel
    # clipLimit controls contrast limiting, tileGridSize controls the area size
    clahe_obj = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(16, 16))
    l_final = clahe_obj.apply(l)

    # 4. Merge the modified L channel with original A and B channels
    l_final_lab = cv2.merge((l_final, a, b))

    # 5. Convert back to RGB space
    image_clahe = cv2.cvtColor(l_final_lab, cv2.COLOR_LAB2RGB)

    return image_clahe


POSTPROCESSING

In [ ]:
from skimage import morphology
import numpy as np

def postprocessing(img):
    """
    Applies morphological post-processing to remove small noise artifacts.
    """
    # 1. Convert to boolean (True where pixel is 255, False where 0)
    # skimage morphology functions require boolean input types
    binary_bool = img.astype(bool)

    # 2. Remove small objects (Morphological cleaning)
    # Removes connected components smaller than 'min_area' pixels.
    # Connectivity=2 means 8-connected neighborhood (diagonal pixels count).
    cleaned_bool = morphology.remove_small_objects(binary_bool, min_size=60, connectivity=2)

    # 3. Convert back to uint8 format (0-255)
    # Multiplies True (1) by 255 to get white pixels
    image_post = cleaned_bool.astype(np.uint8) * 255

    return image_post

In [ ]:
from torchvision import transforms

# Define paths
test_images_dir = os.path.join(current_dir,...) #insert folder name where test set images are saved
output_masks_dir = os.path.join(current_dir,...) #insert folder name where the output masks'll be saved
# Create output folders if not present
os.makedirs(output_masks_dir, exist_ok=True)

# Send model to device and set to eval
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

# Preprocessing: Resize and convert to tensor
preprocess = transforms.Compose([
    transforms.Resize(input_size),
    transforms.ToTensor(),
])

# List test image files (assumes .png/.jpg/.jpeg)
image_files = [f for f in os.listdir(test_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for img_name in image_files:
    img_path = os.path.join(test_images_dir, img_name)
    # Load image
    image = Image.open(img_path).convert('RGB')
    img= np.array(image)
    # Apply all the preprocessing
    mask = ROI_detection(img)
    img_normalized =color_normalization(img,mask)
    img_clahe = clahe(img_normalized, mask)
    filter_img = cv2.GaussianBlur(img_clahe, (3, 3), 0.5)
    image_pre=cv2.bitwise_and(filter_img, filter_img, mask=mask)
    image_pre_pil = Image.fromarray(image_pre)
    input_tensor = preprocess(image_pre_pil).unsqueeze(0).to(device)

    # Inference
    with torch.no_grad():
        output = model(input_tensor)
        pred_mask = torch.sigmoid(output).cpu().squeeze().numpy()
    pred_mask = (pred_mask > 0.5).astype(np.uint8)*255

    # Post-processing
    pred_mask_bin=postprocessing(pred_mask)

    # Save predicted mask
    pred_mask_img = Image.fromarray(pred_mask_bin)
    pred_mask_img.save(os.path.join(output_masks_dir, img_name))


In [ ]:
#METRICS
from sklearn.metrics import matthews_corrcoef
def compute_binary_metrics(manual_mask, auto_mask):

    # Convert masks to boolean arrays
    manual_mask = manual_mask > 0
    auto_mask = auto_mask > 0

    # Check that masks have the same shape
    assert manual_mask.shape == auto_mask.shape, "Masks must have the same shape"

    # True Positives (TP), False Positives (FP), True Negatives (TN), False Negatives (FN)
    TP = np.sum(np.logical_and(manual_mask, auto_mask))
    FP = np.sum(np.logical_and(np.logical_not(manual_mask), auto_mask))
    TN = np.sum(np.logical_and(np.logical_not(manual_mask), np.logical_not(auto_mask)))
    FN = np.sum(np.logical_and(manual_mask, np.logical_not(auto_mask)))


    # Compute common segmentation metrics
    IoU = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0
    Dice = 2 * TP / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 0
    Precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    Recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    Accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
    MCC = matthews_corrcoef(manual_mask.flatten(), auto_mask.flatten())


    return IoU, Dice, Precision, Recall, Accuracy, MCC
